# Часть 1: Преобразование полученных данных в единый формат

In [1]:
import pandas as pd

# Конфиг обработки

- Приводим данные в MS частотность
- Все переменные обозначаем именами из оригинальной статьи
- Проверяем наличие нанов

## CPI US (индекс потребительских цен)

### Нужно получить данные 1972:12 – 2014:12

на один месяц назад больше так как считаем инфляцию

In [18]:
cpi = pd.read_csv("../original_data/CPIAUCSL.csv")

In [19]:
cpi.dtypes

observation_date        str
CPIAUCSL            float64
dtype: object

In [20]:
cpi['date'] = pd.to_datetime(cpi['observation_date'])

In [21]:
cpi.dtypes

observation_date               str
CPIAUCSL                   float64
date                datetime64[us]
dtype: object

In [22]:
pd.infer_freq(cpi['date'])

'MS'

In [23]:
cpi.isna().sum()

observation_date    0
CPIAUCSL            1
date                0
dtype: int64

In [24]:
cpi[cpi['CPIAUCSL'].isna()]

,observation_date,CPIAUCSL,date
945,2025-10-01,NaN,2025-10-01


не страшно

In [25]:
cpi.drop(labels=['observation_date'], axis=1, inplace=True)
cpi.rename(columns={"CPIAUCSL": "cpi_us"}, inplace=True)

In [26]:
cpi = (
    cpi.loc[cpi["date"].between("1972-12-01", "2014-12-01")]
    .copy()
    .reset_index(drop=True)
)
cpi["date"].min(), cpi["date"].max(), len(cpi)

(Timestamp('1972-12-01 00:00:00'), Timestamp('2014-12-01 00:00:00'), 505)

In [27]:
cpi.isna().sum()

cpi_us    0
date      0
dtype: int64

In [42]:
dates = cpi['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [29]:
cpi.to_csv("../original_data/processed/cpi_us.csv", index=False)

## IGREA (Real Global Economic Activity Index `rea_t`)

In [32]:
igrea = pd.read_excel("../original_data/igrea.xlsx")
igrea.dtypes

Date          datetime64[us]
Unnamed: 1           float64
dtype: object

In [33]:
igrea.rename(columns={"Unnamed: 1": "rea_t"}, inplace=True)

In [34]:
igrea.rename(columns={"Date": "date"}, inplace=True)

In [36]:
pd.infer_freq(igrea['date'])

In [38]:
igrea.isna().sum()

date     0
rea_t    0
dtype: int64

In [39]:
# Преобразуем все даты к первому дню месяца
igrea['date'] = pd.to_datetime(igrea['date']).dt.normalize()
igrea['date'] = igrea['date'].dt.to_period('M').dt.start_time

In [40]:
pd.infer_freq(igrea['date'])

'MS'

In [41]:
dates = igrea['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [44]:
igrea = (
    igrea.loc[igrea["date"].between("1973-01-01", "2014-12-01")]
    .copy()
    .reset_index(drop=True)
)
igrea["date"].min(), igrea["date"].max(), len(igrea)

(Timestamp('1973-01-01 00:00:00'), Timestamp('2014-12-01 00:00:00'), 504)

In [45]:
igrea.to_csv("../original_data/processed/igrea.csv", index=False)

## Мировая добыча нефти

In [48]:
oil_world = pd.read_excel("../original_data/oil_prod.xlsx")

In [49]:
oil_world = oil_world.T

In [50]:
oil_world = oil_world.reset_index()

In [51]:
oil_world.drop(labels=['index'], axis=1, inplace=True)

In [52]:
oil_world.columns

Index([0, 1, 2], dtype='object')

In [53]:
print(oil_world.head())
print(oil_world.dtypes)

          0      1       2
0      date  world  russia
1  Jan 1973  54389      --
2  Feb 1973  54930      --
3  Mar 1973  54995      --
4  Apr 1973  55049      --
0    object
1    object
2    object
dtype: object


In [ ]:
# Всё в одном блоке
oil_world.columns = oil_world.iloc[0]
oil_world = oil_world[1:].reset_index(drop=True)
oil_world.columns = ['date', 'world', 'russia']

# Конвертация всех типов
oil_world['world'] = pd.to_numeric(oil_world['world'], errors='coerce')
oil_world['date'] = pd.to_datetime(oil_world['date'], format='%b %Y').dt.to_period('M').dt.start_time

In [56]:
oil_world = oil_world[['date', 'world']]

In [57]:
oil_world.dtypes

date     datetime64[us]
world           float64
dtype: object

In [58]:
pd.infer_freq(oil_world['date'])

'MS'

In [59]:
oil_world.isna().sum()

date     0
world    0
dtype: int64

In [60]:
oil_world = (
    oil_world.loc[oil_world["date"].between("1973-01-01", "2014-12-01")]
    .copy()
    .reset_index(drop=True)
)
oil_world["date"].min(), oil_world["date"].max(), len(oil_world)

(Timestamp('1973-01-01 00:00:00'), Timestamp('2014-12-01 00:00:00'), 504)

In [61]:
oil_world.rename(columns={"world": "oil_prod_world"}, inplace=True)

In [62]:
oil_world

,date,oil_prod_world
0,1973-01-01,54389.000000
1,1973-02-01,54930.000000
2,1973-03-01,54995.000000
3,1973-04-01,55049.000000
4,1973-05-01,56323.000000
...,...,...
499,2014-08-01,78491.319413
500,2014-09-01,79227.607413
501,2014-10-01,80212.039413
502,2014-11-01,79975.463413


In [63]:
dates = oil_world['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [64]:
oil_world.to_csv("../original_data/processed/oil_prod_world.csv", index=False)

## The nominal price of oil

In [67]:
oil_price = pd.read_csv("../original_data/U.S._Crude_Oil_Imported_Acquisition_Cost_by_Refiners.csv")

In [68]:
oil_price.dtypes

date         str
npo_t    float64
dtype: object

In [69]:
print(oil_price.head())

       date  npo_t
0  Apr 2026  94.03
1  Mar 2026  84.71
2  Feb 2026  61.43
3  Jan 2026  59.05
4  Dec 2025  57.26


In [72]:
oil_price["date"] = pd.to_datetime(oil_price["date"], format="%b %Y")

In [75]:
oil_price.isna().sum()

date     0
npo_t    0
dtype: int64

In [77]:
oil_price = oil_price.sort_values(by='date')

In [78]:
dates = oil_price['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [79]:
pd.infer_freq(oil_price['date'])

'MS'

In [80]:
oil_price = (
    oil_price.loc[oil_price["date"].between("1973-01-01", "2014-12-01")]
    .copy()
    .reset_index(drop=True)
)
oil_price["date"].min(), oil_price["date"].max(), len(oil_world)

(Timestamp('1974-01-01 00:00:00'), Timestamp('2014-12-01 00:00:00'), 504)

In [81]:
oil_price.to_csv("../original_data/processed/nominal_oil_price.csv", index=False)

## The aggregate U.S. nominal stock market return (`net_t`)

In [123]:
stocks = pd.read_csv("../original_data/F-F_Research_Data_Factors.csv")

In [124]:
stocks = stocks[['date', 'Mkt-RF', 'RF']]

In [125]:
stocks.dtypes

date        int64
Mkt-RF    float64
RF        float64
dtype: object

In [126]:
print(stocks.head())

     date  Mkt-RF    RF
0  192607    2.89  0.22
1  192608    2.64  0.25
2  192609    0.38  0.23
3  192610   -3.27  0.32
4  192611    2.54  0.31


In [127]:
stocks["date"] = pd.to_datetime(stocks["date"].astype(str), format="%Y%m")

In [128]:
pd.infer_freq(stocks['date'])

'MS'

In [129]:
dates = stocks['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [130]:
stocks = (
    stocks.loc[stocks["date"].between("1973-01-01", "2014-12-01")]
    .copy()
    .reset_index(drop=True)
)
stocks["date"].min(), stocks["date"].max(), len(stocks)

(Timestamp('1973-01-01 00:00:00'), Timestamp('2014-12-01 00:00:00'), 504)

In [131]:
stocks.isna().sum()

date      0
Mkt-RF    0
RF        0
dtype: int64

- RF - это безрисковая ставка
- Mkt-RF - это рыночная доходность минус безрисковая ставка

---
Почему вообще эти данные подходят?

В оригинальной статье пишут "The aggregate U.S. real stock market return is obtained by subtracting the CPI inflation rate from the log returns on the CRSP value-weighted market portfolio. "

Здесь заложено два ключевых шага из оригинальной статьи Killian & Park:
- получить лог-доходность (log returns) CRSP value-weighted market portfolio
- вычесть из нее уровень инфляции (CPI inflation rate), чтобы перейти от номинальной лог-доходности к реальной

Данные с сайта Кена Френча — это самый простой и точный способ получить прокси для CRSP value-weighted market portfolio.

Mkt-RF (Rm-Rf): Это избыточная доходность рынка. Она рассчитывается именно как log return on CRSP value-weighted market portfolio - risk-free rate (RF).

RF (Risk-free rate): Это безрисковая ставка, обычно доходность 1-месячных казначейских векселей США, которая уже представлена в виде лог-доходности

поэтому для получения искомой доходности надо просто сложить полученные `mkt-rf` + `rf`

на постобработке вычтем инфляцию и получим исходные данные

In [135]:
print(stocks.iloc[65:80])

         date  Mkt-RF    RF
65 1978-06-01   -1.69  0.54
66 1978-07-01    5.12  0.56
67 1978-08-01    3.78  0.56
68 1978-09-01   -1.44  0.62
69 1978-10-01  -11.90  0.68
70 1978-11-01    2.71  0.70
71 1978-12-01    0.87  0.78
72 1979-01-01    4.22  0.77
73 1979-02-01   -3.56  0.73
74 1979-03-01    5.67  0.81
75 1979-04-01   -0.07  0.80
76 1979-05-01   -2.20  0.82
77 1979-06-01    3.84  0.81
78 1979-07-01    0.83  0.77
79 1979-08-01    5.54  0.77


In [137]:
stocks['stock_market_return'] = stocks['Mkt-RF'] + stocks['RF']

In [140]:
stocks = stocks[['date', 'stock_market_return']]

In [141]:
stocks.to_csv(
    "../original_data/processed/stock_market_return.csv",
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.2f",
)

## Добыча нефти в США (US oil production)

In [98]:
oil_usa = pd.read_excel("../original_data/usa_oil_prod.xlsx")

In [100]:
oil_usa = oil_usa.T

In [101]:
oil_usa

,0
date,oil_prod_usa
Jan 1973,9175.935
Feb 1973,9395.214
Mar 1973,9271.935
Apr 1973,9291-09-01 00:00:00
...,...
Oct 2025,13863.763
Nov 2025,13789.249
Dec 2025,13656.661
Jan 2026,13237.136


In [102]:
oil_usa = oil_usa.reset_index()

In [103]:
oil_usa

,index,0
0,date,oil_prod_usa
1,Jan 1973,9175.935
2,Feb 1973,9395.214
3,Mar 1973,9271.935
4,Apr 1973,9291-09-01 00:00:00
...,...,...
634,Oct 2025,13863.763
635,Nov 2025,13789.249
636,Dec 2025,13656.661
637,Jan 2026,13237.136


In [105]:
oil_usa.columns = oil_usa.iloc[0]

In [107]:
oil_usa = oil_usa[1:].reset_index(drop=True)

In [110]:
print(oil_usa)

0        date         oil_prod_usa
0    Jan 1973             9175.935
1    Feb 1973             9395.214
2    Mar 1973             9271.935
3    Apr 1973  9291-09-01 00:00:00
4    May 1973             9262.387
..        ...                  ...
633  Oct 2025            13863.763
634  Nov 2025            13789.249
635  Dec 2025            13656.661
636  Jan 2026            13237.136
637  Feb 2026            13625.537

[638 rows x 2 columns]


In [112]:
from datetime import date, datetime
import re

def fix_excel_date_number(x):
    if isinstance(x, (datetime, date)):
        return x.year + x.month / 10

    if isinstance(x, str):
        match = re.fullmatch(r"(\d{4})-(\d{2})-\d{2}(?: 00:00:00)?", x)
        if match:
            year = int(match.group(1))
            month = int(match.group(2))
            return year + month / 10

    return x

In [114]:
oil_usa["oil_prod_usa"] = (
    oil_usa["oil_prod_usa"]
    .apply(fix_excel_date_number)
    .astype(float)
)

In [115]:
oil_usa.dtypes

0
date                str
oil_prod_usa    float64
dtype: object

In [116]:
oil_usa["date"] = pd.to_datetime(oil_usa["date"], format="%b %Y")

In [118]:
pd.infer_freq(oil_usa['date'])

'MS'

In [119]:
oil_usa = (
    oil_usa.loc[oil_usa["date"].between("1973-01-01", "2014-12-01")]
    .copy()
    .reset_index(drop=True)
)
oil_usa["date"].min(), oil_usa["date"].max(), len(oil_usa)

(Timestamp('1973-01-01 00:00:00'), Timestamp('2014-12-01 00:00:00'), 504)

In [120]:
oil_usa.isna().sum()

0
date            0
oil_prod_usa    0
dtype: int64

In [121]:
dates = oil_usa['date'].tolist()
for i in range(len(dates)-1):
    assert dates[i+1] == pd.date_range(start=dates[i], periods=2, freq="MS")[1]

In [122]:
oil_usa.to_csv("../original_data/processed/oil_prod_usa.csv", index=False)